In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! pip install pytorch-transformers
! pip install transformers
!pip install pymorphy2
!pip install -U pymorphy2-dicts-ru
!pip install -U pymorphy2-dicts-uk

In [ ]:
import torch
from pytorch_transformers import AdamW, BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras.preprocessing.sequence import pad_sequences
from pytorch_transformers import BertTokenizer, BertConfig
from tqdm import tqdm, trange
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import json
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from transformers import BertForSequenceClassification, AutoModelForSequenceClassification
from transformers import BertTokenizer, BertConfig, AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup
import re
from IPython.display import clear_output


import pymorphy2
morph = pymorphy2.MorphAnalyzer()

In [ ]:
config = "DeepPavlov/bert-base-bg-cs-pl-ru-cased"
MODEL_SAVE_NAME = "/content/drive/MyDrive/onti/slavik_pretrain_classifier"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def preprocessing(data):
  prep_data = []
  entities = []
  querys = []
  labels = []

  for i, sample in enumerate(data):
    context = sample['passage']['text']
    prepared_data.append(context)

    variants = []
    for e in sample['passage']['entities']:
      end = e['end']
      start = e['start']
      variants.append(context[start:end])
    entities.append(tuple(set(variants)))

    querys.append(sample['qas'][0]['query'])

    if 'answers' in sample['qas'][0].keys():
      answers = []
      for answer in sample['qas'][0]['answers']:
        answers.append(answer['text'])
      labels.append(tuple(set(answers)))
  
  return [prep_data, entities, querys, labels]

In [ ]:
def dataloader(data, batch_size=8):
  # Объединение
  sentences = []
  labels = []
  for minibatch in zip(*data):
    prep_data, entities, querys, answers = minibatch
    prep_data = re.sub('@header|@highlight|@context', '', prep_data)
    context_texts = prep_data.split('.')
    for ent in entities:
      context = ''
      for text in context_texts:
        if text.find(ent) > 0:
          context += ' ' + text

      p = morph.parse(ent)[0]
      tag = str(p.tag)

          
      sub_query = re.sub('@placeholder', '[MASK]', querys)
      sentences.append("[CLS] " + sub_query + " [SEP] " + ent + " [SEP] " + tag + " [SEP] " + context + " [SEP]")
      labels.append([int(ent in answers)]) ####
  sentences = np.array(sentences)
  labels = np.array(labels)
  
  # Batch Normalization 
  zero_mask = labels[:, 0] == 0
  ones_mask = labels[:, 0] == 1
  choices = np.random.choice(zero_mask.sum(), ones_mask.sum())
  sentences_balanced = np.hstack((sentences[ones_mask], sentences[zero_mask][choices]))
  labels_balanced = np.vstack((labels[ones_mask], labels[zero_mask][choices]))


  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
  tokenized_texts = [tokenizer.tokenize(sent) for sent in sentences_balanced]
  input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]

  input_ids = pad_sequences(
      input_ids,
      maxlen=512,
      dtype="long",
      truncating="post",
      padding="post"
      ) 

  attention_masks = [[float(i>0) for i in seq] for seq in input_ids]

  train_inputs = torch.tensor(input_ids)
  train_labels = torch.tensor(labels_balanced)
  train_masks = torch.tensor(attention_masks)

  train_data = TensorDataset(train_inputs, train_masks, train_labels)
  dataloader = DataLoader(
      train_data,
      sampler=RandomSampler(train_data),
      batch_size=batch_size
  )

  return dataloader

In [ ]:
def double_exponential_smoothing(series, alpha, beta):
    result = [series[0]]
    for n in range(1, len(series)+1):
        if n == 1:
            level, trend = series[0], series[1] - series[0]
        if n >= len(series): # прогнозируем
            value = result[-1]
        else:
            value = series[n]
        last_level, level = level, alpha*value + (1-alpha)*(level+trend)
        trend = beta*(level-last_level) + (1-beta)*trend
        result.append(level+trend)
    return result

In [ ]:
def freeze_model(model, mode, name_param=None):
  if mode:
    for name, module in model.named_parameters():
      if name_param not in name:
        module.requires_grad = False
      else:
        module.requires_grad = True
  else:
    for name, module in model.named_parameters():
      module.requires_grad = True

In [ ]:
with open("/content/drive/MyDrive/onti/RuCoS/rucos_train.jsonl", "r") as read_file:
    json_list_train = list(read_file)
train_data = [json.loads(line) for line in json_list_train]

with open("/content/drive/MyDrive/onti/RuCoS/rucos_val.jsonl", "r") as read_file:
    json_list_val = list(read_file)
val_data = [json.loads(line) for line in json_list_val]

with open("/content/drive/MyDrive/onti/RuCoS/rucos_test.jsonl", "r") as read_file:
    json_list_test = list(read_file)
test_data = [json.loads(line) for line in json_list_test]

In [ ]:
p_train_data = preprocessing(train_data)
p_val_data = preprocessing(val_data)
p_test_data = preprocessing(test_data)
p_train_val = preprocessing(train_data+val_data)

In [ ]:
validation_dataloader = dataloader(p_val_data)

In [ ]:
train_val_dataloader = dataloader(p_train_val)

In [ ]:
model = BertForSequenceClassification.from_pretrained(config, num_labels=2)
model.to(device)

In [ ]:
param_optimizer = list(model.named_parameters())

no_decay = ['bias', 'gamma', 'beta']
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
     'weight_decay_rate': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
     'weight_decay_rate': 0.0}
]

optimizer = AdamW(optimizer_grouped_parameters, lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_optim_steps)

In [ ]:
num_optim_steps = int(len(train_val_dataloader)*8)
num_warmup_steps = 500
SAVE_STEP = 1000
PRETRAIN_STEP = 1000



train_loss_set = []
train_loss = 0


freeze_model(model, 1, 'classifier')
for epoch in range(3):
  model.train()
  for step ,batch in enumerate(train_val_dataloader):
      if (step+1) % SAVE_STEP == 0:
        model.save_pretrained(MODEL_SAVE_NAME)
      if step == PRETRAIN_STEP:
        freeze_model(model, 0)
 
      batch = tuple(t.to(device) for t in batch)
    
      b_input_ids, b_input_mask, b_labels = batch
      
      # если не сделать .zero_grad(), градиенты будут накапливаться
      optimizer.zero_grad()
      
   
      loss = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
      train_loss_set.append(loss[0].item())  
      
   
      loss[0].backward()
      
    
      optimizer.step()
      scheduler.step()


      train_loss += loss[0].item()
      
      if(step+1)%100 == 0:
      
        clear_output(True)
        smoothing_train_loss_set = double_exponential_smoothing(train_loss_set, 0.05, 0.05)
        plt.plot(smoothing_train_loss_set)
        plt.title("Training loss")
        plt.xlabel("Batch")
        plt.ylabel("Loss")
        plt.show()
      
  print("Loss на обучении: {0:.5f}".format(train_loss / len(all_dataloader)*(epoch+1)))

In [ ]:
model.save_pretrained(MODEL_SAVE_NAME)

In [ ]:
def predict_on_batch(model, b_input_ids, b_input_mask):
  with torch.no_grad():
    logits = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
    logits = logits[0].detach().cpu().numpy()
    return logits.tolist()

In [ ]:
def validation(data, model, batch_size=8):
  model.eval()
  output = []
  tokenizer = AutoTokenizer.from_pretrained(config, do_lower_case=False)

  for step, minibatch in enumerate(tqdm(zip(*data))):
    pred_logits = []
    b_input_ids = []
    b_input_mask = []

    prep_data, entities, querys, labels = minibatch
    prep_data = re.sub('@header|@highlight|@context', '', prep_data)
    context_texts = prep_data.split('.')

    for n_ent, ent in enumerate(entities):
        context = ''
        for text in context_texts:
          if text.find(ent) > 0:
            context += ' ' + text

        p = morph.parse(ent)[0]
        tag = str(p.tag)

        sub_query = re.sub('@placeholder', '[MASK]', querys)
        sentence = "[CLS] " + sub_query + " [SEP] " + ent + " [SEP] " + tag + " [SEP] " + context + " [SEP]"

        tokenized_text = tokenizer.tokenize(sentence)
        input_ids = tokenizer.convert_tokens_to_ids(tokenized_text)
        input_ids = pad_sequences(
            [input_ids, ],
            maxlen=512,
            dtype="long",
            truncating="post",
            padding="post"
        )
        attention_mask = [float(j>0) for j in input_ids[0]]

        b_input_ids.append(input_ids[0])
        b_input_mask.append(attention_mask)

        if ((n_ent+1) == batch_size) or (n_ent == (len(entities)-1)):
          b_input_ids_tensor = torch.tensor(b_input_ids).to(device)
          b_input_mask_tensor = torch.tensor(b_input_mask).to(device)
          pred_logits.extend(predict_on_batch(model, b_input_ids_tensor, b_input_mask_tensor))

          b_input_ids = []
          b_input_mask = []
    
    n_max_confidence = np.argmax(np.array(pred_logits)[:, 1])
    if entities[n_max_confidence] in labels:
      output.append(1)
    else:
      output.append(0)
  
  valid = [1 for i in range(len(output))]

  print("Процент правильных предсказаний на валидационной выборке: {0:.2f}%".format(
      accuracy_score(valid, output) * 100))
  print("F1 score на валидационной выборке: {0:.2f}".format(
      f1_score(valid, output, average='weighted')))

In [ ]:
validation(prepare_val_data, model)

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_SAVE_NAME, num_labels=2)
model.to(device)

In [ ]:
def submit(file_name, model, batch_size=8):
  model.eval()
  output = []
  tokenizer = AutoTokenizer.from_pretrained(config, do_lower_case=False)

  for step, minibatch in enumerate(tqdm(zip(*p_test_data[:-1]))):
    pred_logits = []
    b_input_ids = []
    b_input_mask = []

    prep_data, entities, querys = minibatch
    prep_data = re.sub('@header|@highlight|@context', '', prep_data)
    context_texts = prep_data.split('.')

    for n_ent, ent in enumerate(entities):
        context = ''
        for text in context_texts:
          if text.find(ent) > 0:
            context += ' ' + text

        p = morph.parse(ent)[0]
        tag = str(p.tag)

        sub_query = re.sub('@placeholder', '[MASK]', querys)
        sentence = "[CLS] " + sub_query + " [SEP] " + ent + " [SEP] " + tag + " [SEP] " + context + " [SEP]"

        tokenized_text = tokenizer.tokenize(sentence)
        input_ids = tokenizer.convert_tokens_to_ids(tokenized_text)
        input_ids = pad_sequences(
            [input_ids, ],
            maxlen=512,
            dtype="long",
            truncating="post",
            padding="post"
        )
        attention_mask = [float(j>0) for j in input_ids[0]]

        b_input_ids.append(input_ids[0])
        b_input_mask.append(attention_mask)

        if ((n_ent+1) == batch_size) or (n_ent == (len(entities)-1)):
          b_input_ids_tensor = torch.tensor(b_input_ids).to(device)
          b_input_mask_tensor = torch.tensor(b_input_mask).to(device)
          pred_logits.extend(predict_on_batch(model, b_input_ids_tensor, b_input_mask_tensor))

          b_input_ids = []
          b_input_mask = []
    
    n_max_confidence = np.argmax(np.array(pred_logits)[:, 1])
    start = prepared_data.find(entities[n_max_confidence])
    end = start + len(entities[n_max_confidence])
    out_dict = {"idx": step,
                "end": end,
                "start": start,
                "text": entities[n_max_confidence]}
    output.append(out_dict)

  with open(file_name, "w") as write_file:
    for out in output:
      json.dump(out, write_file, ensure_ascii=False)
      write_file.write('\n')

In [ ]:
submit("/content/drive/MyDrive/onti/slavik_2.jsonl", model)

In [ ]:
def submit_logits(file_name, model, batch_size=8):
  model.eval()
  output = []
  tokenizer = AutoTokenizer.from_pretrained(config, do_lower_case=False)

  for step, minibatch in enumerate(tqdm(zip(*p_test_data[:-1]))):
    pred_logits = []
    b_input_ids = []
    b_input_mask = []

    prep_data, entities, querys = minibatch
    prep_data = re.sub('@header|@highlight|@context', '', prep_data)
    context_texts = prep_data.split('.')

    for n_ent, ent in enumerate(entities):
        context = ''
        for text in context_texts:
          if text.find(ent) > 0:
            context += ' ' + text


        p = morph.parse(ent)[0]
        tag = str(p.tag)
        

        sub_query = re.sub('@placeholder', '[MASK]', querys)
        sentence = "[CLS] " + context + sub_query + " [SEP] " + ent + " [SEP] " + tag + " [SEP]"
        tokenized_text = tokenizer.tokenize(sentence)
        input_ids = tokenizer.convert_tokens_to_ids(tokenized_text)
        input_ids = pad_sequences(
            [input_ids, ],
            maxlen=512,
            dtype="long",
            truncating="post",
            padding="post"
        )
        attention_mask = [float(j>0) for j in input_ids[0]]

        b_input_ids.append(input_ids[0])
        b_input_mask.append(attention_mask)

        if ((n_ent+1) == batch_size) or (n_ent == (len(entities)-1)):
          b_input_ids_tensor = torch.tensor(b_input_ids).to(device)
          b_input_mask_tensor = torch.tensor(b_input_mask).to(device)
          pred_logits.extend(predict_on_batch(model, b_input_ids_tensor, b_input_mask_tensor))

          b_input_ids = []
          b_input_mask = []

    out_dict = {"idx": step,
                "logits": pred_logits,
                'entities': entities}
    output.append(out_dict)

  with open(file_name, "w") as write_file:
    for out in output:
      json.dump(out, write_file, ensure_ascii=False)
      write_file.write('\n')

In [ ]:
submit_logits("/content/drive/MyDrive/onti/blend2.jsonl", model)